# SpaceX Falcon 9 | Historical Data Acquisition & Quality Audit

**Notebook 01 · IBM Applied Data Science Capstone — independently modernised portfolio edition**

### Purpose
Establish a **reproducible, auditable historical launch dataset** for later exploratory analysis and landing-outcome modelling. This notebook retrieves the IBM-hosted `dataset_part_1.csv`, preserves its source bytes, validates its schema and values, and exports a lightly standardised copy with a machine-readable audit report.

**Data provenance:** [IBM Skills Network dataset](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_1.csv). The source is an **already-processed course dataset**, not a fresh extraction of raw SpaceX API JSON. No claim of live API collection is made. The original course notebook remains in `original-coursework/`.

**Research question for the broader project:** Which mission and booster characteristics are associated with Falcon 9 first-stage landing outcomes? This notebook **does not** estimate those associations or train a model.

**Expected source at time of development:** 90 rows and 17 columns, as verified in the user's local run. The notebook checks actual source content rather than hard-coding those dimensions as a guarantee.

## 1 · Environment and file locations

Run from the repository root (or change `PROJECT_ROOT`). Requires Python 3.10+, `pandas`, `requests`, `IPython` and `matplotlib` only for the optional chart. This notebook does not require the original SpaceX API to be online.

**Offline-first behaviour:** after the first successful download, the saved CSV is reused. Set `REFRESH_SOURCE=True` to download a new copy deliberately; the existing copy is not overwritten if a download fails.

In [1]:
from __future__ import annotations

from datetime import datetime, timezone
from io import BytesIO
from pathlib import Path
import hashlib
import json

import pandas as pd
import requests
from IPython.display import display

PROJECT_ROOT = Path.cwd()  # Change if you launch Jupyter outside the repository root.
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

DATA_URL = (
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/"
    "IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_1.csv"
)
RAW_CSV = RAW_DIR / "ibm_spacex_dataset_part_1.csv"
SOURCE_METADATA = RAW_DIR / "ibm_spacex_dataset_part_1_metadata.json"
OUTPUT_CSV = PROCESSED_DIR / "falcon9_launches_historical.csv"
QUALITY_JSON = PROCESSED_DIR / "falcon9_data_quality.json"
REFRESH_SOURCE = False
REQUEST_TIMEOUT = 30

print("Project root:", PROJECT_ROOT)

Project root: c:\Users\kanwa\Downloads\New folder\notebooks


## 2 · Acquire and preserve the source

The downloaded bytes are stored unchanged, and a SHA-256 digest identifies the exact source file used. If a local copy already exists, no network connection is required. The metadata records the original download time when available; a later offline run does not pretend to have downloaded the file again.

In [2]:
def download_source(url: str) -> bytes:
    try:
        response = requests.get(url, timeout=REQUEST_TIMEOUT)
        response.raise_for_status()
    except requests.RequestException as exc:
        raise RuntimeError(
            f"Unable to download IBM historical CSV: {url}. "
            "If you previously downloaded it, place the CSV at "
            f"{RAW_CSV} and rerun."
        ) from exc
    content = response.content
    if not content or b"," not in content[:1000]:
        raise ValueError("Downloaded response does not appear to contain a CSV file")
    return content

if RAW_CSV.exists() and not REFRESH_SOURCE:
    raw_bytes = RAW_CSV.read_bytes()
    source_mode = "cached local copy"
    if SOURCE_METADATA.exists():
        source_metadata = json.loads(SOURCE_METADATA.read_text(encoding="utf-8"))
    else:
        source_metadata = {"downloaded_at_utc": None}
else:
    raw_bytes = download_source(DATA_URL)
    # Verify parseability before replacing a previously cached dataset.
    pd.read_csv(BytesIO(raw_bytes))
    RAW_CSV.write_bytes(raw_bytes)
    source_metadata = {"downloaded_at_utc": datetime.now(timezone.utc).isoformat()}
    SOURCE_METADATA.write_text(json.dumps(source_metadata, indent=2), encoding="utf-8")
    source_mode = "new download"

source_sha256 = hashlib.sha256(raw_bytes).hexdigest()
raw_df = pd.read_csv(BytesIO(raw_bytes))
print("Source:", source_mode)
print("Rows × columns:", raw_df.shape)
print("SHA-256:", source_sha256)
display(raw_df.head())

Source: new download
Rows × columns: (90, 17)
SHA-256: 606fecaf0209f41cc79aeb383b8519ba854bab8b198762952bf0c3c34e0c6c6f


,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude
0,1,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857
3,4,2013-09-29,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1,False,False,False,NaN,1.0,0,B1003,-120.610829,34.632093
4,5,2013-12-03,Falcon 9,3170.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857


## 3 · Validate the dataset contract

We require columns needed by downstream notebooks and report optional columns as actually supplied. The source's original row order and flight numbers are preserved. The notebook fails clearly if the source structure changes rather than silently creating a misleading export.

In [3]:
REQUIRED_COLUMNS = {
    "FlightNumber", "Date", "BoosterVersion", "PayloadMass", "Orbit", "LaunchSite"
}
missing_required = sorted(REQUIRED_COLUMNS - set(raw_df.columns))
if missing_required:
    raise ValueError(f"Source dataset missing required columns: {missing_required}")
if raw_df.empty:
    raise ValueError("Source dataset has zero rows")
if raw_df.columns.duplicated().any():
    raise ValueError("Source dataset has duplicate column names")

launch_df = raw_df.copy()
launch_df["Date"] = pd.to_datetime(launch_df["Date"], errors="coerce")
if launch_df["Date"].isna().any():
    raise ValueError("Invalid or missing dates; inspect the raw CSV before proceeding")
if launch_df["FlightNumber"].isna().any():
    raise ValueError("Missing flight numbers")

launch_df["PayloadMass"] = pd.to_numeric(launch_df["PayloadMass"], errors="coerce")
if (launch_df["PayloadMass"].dropna() < 0).any():
    raise ValueError("Negative payload masses detected")

# Avoid silently claiming every rocket label is exactly 'Falcon 9':
# course data often contains booster variants such as 'Falcon 9 Block 5'.
non_falcon9 = launch_df.loc[
    ~launch_df["BoosterVersion"].astype("string").str.startswith("Falcon 9", na=False),
    "BoosterVersion"
].drop_duplicates().tolist()
if non_falcon9:
    raise ValueError(f"Unexpected non-Falcon-9 booster labels: {non_falcon9}")

print("Schema validation passed")
print("Date range:", launch_df["Date"].min().date(), "to", launch_df["Date"].max().date())
print("Column names:", list(launch_df.columns))
display(launch_df.dtypes.astype(str).rename("dtype").to_frame())

Schema validation passed
Date range: 2010-06-04 to 2020-11-05
Column names: ['FlightNumber', 'Date', 'BoosterVersion', 'PayloadMass', 'Orbit', 'LaunchSite', 'Outcome', 'Flights', 'GridFins', 'Reused', 'Legs', 'LandingPad', 'Block', 'ReusedCount', 'Serial', 'Longitude', 'Latitude']


,dtype
FlightNumber,int64
Date,datetime64[ns]
BoosterVersion,object
PayloadMass,float64
Orbit,object
LaunchSite,object
Outcome,object
Flights,int64
GridFins,bool
Reused,bool


## 4 · Quality assessment

Missing values are reported rather than silently filled. **No payload-mass imputation** is performed: any modelling imputer must be fitted to training data only. Duplicate rows and duplicate flight numbers are reported separately; the latter is not automatically treated as an error without examining the source semantics.

The original course dataset may contain an `Outcome` text field rather than a ready-made binary `LandingSuccess` label. The next notebook will define the modelling target explicitly.

In [4]:
missing_counts = launch_df.isna().sum().sort_values(ascending=False)
missing_percent = (100 * missing_counts / len(launch_df)).round(1)
missing_table = pd.DataFrame({"missing_count": missing_counts, "missing_pct": missing_percent})

quality_report = {
    "source_url": DATA_URL,
    "source_type": "IBM-hosted processed historical course CSV",
    "source_sha256": source_sha256,
    "source_mode": source_mode,
    "downloaded_at_utc": source_metadata.get("downloaded_at_utc"),
    "audited_at_utc": datetime.now(timezone.utc).isoformat(),
    "rows": int(len(launch_df)),
    "columns": int(len(launch_df.columns)),
    "column_names": list(launch_df.columns),
    "date_min": str(launch_df["Date"].min().date()),
    "date_max": str(launch_df["Date"].max().date()),
    "missing_values_by_column": {k: int(v) for k, v in missing_counts.items()},
    "duplicate_full_rows": int(launch_df.duplicated().sum()),
    "duplicate_flight_numbers": int(launch_df["FlightNumber"].duplicated().sum()),
    "payload_mass_imputed": False,
    "rows_removed": 0,
}

display(missing_table)
print("Duplicate full rows:", quality_report["duplicate_full_rows"])
print("Duplicate flight numbers:", quality_report["duplicate_flight_numbers"])
print("Missing payload mass:", quality_report["missing_values_by_column"]["PayloadMass"])

,missing_count,missing_pct
LandingPad,26,28.9
FlightNumber,0,0.0
Date,0,0.0
PayloadMass,0,0.0
BoosterVersion,0,0.0
LaunchSite,0,0.0
Outcome,0,0.0
Flights,0,0.0
Orbit,0,0.0
GridFins,0,0.0


Duplicate full rows: 0
Duplicate flight numbers: 0
Missing payload mass: 0


## 5 · Small exploratory sanity checks

These descriptive summaries are for data inspection only. They are not evidence of causal relationships or model performance.

In [5]:
print("Booster versions:")
display(launch_df["BoosterVersion"].value_counts(dropna=False).rename("launches").to_frame())
print("Launch sites:")
display(launch_df["LaunchSite"].value_counts(dropna=False).rename("launches").to_frame())
if "Outcome" in launch_df.columns:
    print("Recorded landing outcomes (source labels):")
    display(launch_df["Outcome"].value_counts(dropna=False).rename("launches").to_frame())

Booster versions:


,launches
BoosterVersion,
Falcon 9,90


Launch sites:


,launches
LaunchSite,
CCAFS SLC 40,55
KSC LC 39A,22
VAFB SLC 4E,13


Recorded landing outcomes (source labels):


,launches
Outcome,
True ASDS,41
None None,19
True RTLS,14
False ASDS,6
True Ocean,5
False Ocean,2
None ASDS,2
False RTLS,1


## 6 · Export validated historical data

Exports use ISO-formatted dates and preserve all other source columns. The raw CSV remains unchanged. A separate JSON audit captures provenance, missingness and basic integrity checks. Later notebooks should read the processed CSV and cite the original IBM dataset.

In [6]:
export_df = launch_df.copy()
export_df["Date"] = export_df["Date"].dt.strftime("%Y-%m-%d")
export_df.to_csv(OUTPUT_CSV, index=False)
QUALITY_JSON.write_text(json.dumps(quality_report, indent=2), encoding="utf-8")

# Confirm the saved file is readable and retains its expected dimensions.
reloaded = pd.read_csv(OUTPUT_CSV)
assert reloaded.shape == export_df.shape, "Export dimensions changed unexpectedly"
assert reloaded.columns.tolist() == export_df.columns.tolist(), "Export schema changed unexpectedly"
print("Exported:", OUTPUT_CSV)
print("Quality report:", QUALITY_JSON)
print("Validated export shape:", reloaded.shape)

Exported: c:\Users\kanwa\Downloads\New folder\notebooks\data\processed\falcon9_launches_historical.csv
Quality report: c:\Users\kanwa\Downloads\New folder\notebooks\data\processed\falcon9_data_quality.json
Validated export shape: (90, 17)


## 7 · Limitations and next steps

1. **Historical, not current:** the file covers the period represented by IBM's course snapshot; it should not be presented as current SpaceX launch data.
2. **Processed source:** IBM's course transformations preceded this notebook. We can audit the supplied table but cannot reconstruct every upstream API decision from the CSV alone.
3. **Selection effects:** the course dataset may omit mission configurations or records; its coverage must not be described as all SpaceX launches without separate verification.
4. **Unknown labels and missingness:** missing payload mass is preserved. Landing outcome labels require explicit definition and validation before modelling.
5. **Reproducibility:** commit the notebook and (if redistribution is appropriate) the processed data; record the source URL and SHA-256. If the IBM endpoint disappears, an existing cached CSV supports offline execution.

**Next:** `02_spacex_data_wrangling.ipynb` — inspect outcome semantics, derive a defensible binary target, assess missingness and prepare downstream analysis.

**Attribution:** Based on an IBM Skills Network dataset used in the Applied Data Science Capstone. Original educational materials are retained separately; this refactoring and audit structure are independent portfolio work.